# 08_06_制約デコーディングを使う_自分で書く

コード：`ch08/blocks/08_06_制約デコーディングを使う_自分で書く.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

モデル・データの取得、または長い計算を含みます。

この例はローカルDocker用です。Colab管理ランタイムでの実行対象にはしていません。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


## 本文の掲載コード（参照）


```python
# --- vLLM 0.33（サーバ運用） ---
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
llm = LLM(model="Qwen/Qwen3-8B",
          structured_outputs_config={"backend": "xgrammar",
                                     "disable_any_whitespace": True})  # ★ JSON の空白を禁じる
out = llm.generate(prompt, SamplingParams(
    temperature=0.2, max_tokens=512,
    structured_outputs=StructuredOutputsParams(json=Ticket.model_json_schema())))
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import shutil
import subprocess

if shutil.which("docker") is None:
    raise RuntimeError("この例はDockerの動くローカル環境で実行してください")
(CODE / "ch08/outputs").mkdir(parents=True, exist_ok=True)
subprocess.run(["sh", str(CODE / "ch08/run_vllm.sh"), "Qwen/Qwen2.5-0.5B-Instruct"], cwd=CODE, check=True)
